# 06 — Backward Pass and Gradients

**Purpose:** Show how TorchLens captures the autograd backward pass: gradient fields on Op,
the GradFn/GradFnCall record layer, backward-parity validation, and the backward graph render.

**Surfaces covered:**
- [ ] `tl.trace(..., backward_ready=True, save_grads=True)` — backward-enabled capture
- [ ] `trace.backward(loss)` — deferred backward capture on a Trace
- [ ] `Op.grad` / `Op.has_grad` / `Op.has_saved_gradient` — gradient fields on Op
- [ ] `Op.grad_fn_class_name` / `Op.grad_fn_label` — autograd linkage fields on Op
- [ ] `trace.grad_fns` / `trace.saved_grad_fns` — GradFn accessor
- [ ] `GradFn.__repr__` — record repr
- [ ] `GradFn.calls` / `GradFnCall.__repr__` — per-fire records + reprs
- [ ] `GradFnCall.grad_inputs` / `GradFnCall.grad_outputs` — payload tensors on call
- [ ] `trace.num_grad_fns` / `trace.num_saved_grad_fns` — summary counts
- [ ] `tl.validate(model, x, scope='backward', loss_fn=...)` — backward parity check
- [ ] `tl.validate(model, x, scope='forward')` / `scope='saved'` — other scopes
- [ ] `trace.draw_backward(...)` — backward graph smoke render

## 1. Environment setup

In [ ]:
import pathlib
import sys
import tempfile
import os

sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torch.nn as nn
import torchlens as tl
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

## 2. Backward-enabled capture

Two kwargs gate backward support:
- `backward_ready=True` — keeps the autograd graph alive after forward; required for `trace.backward()`
- `save_grads=True` — saves gradient tensors onto Op records during backward

The `linear_relu` zoo model (`fc -> relu -> sum`) produces a scalar loss, making `.backward()` trivial.

In [ ]:
# linear_relu: Linear(3,4) -> relu -> sum  (scalar output => easy backward)
model, x = ZOO["linear_relu"]()

trace = tl.trace(model, x, backward_ready=True, save_grads=True)

print("repr:", repr(trace))
print()
print("backward_ready :", trace.backward_ready)
print("has_gradients  :", trace.has_gradients)  # False — backward not run yet
print("has_backward_pass:", trace.has_backward_pass)

## 3. Running `trace.backward(loss)`

Pass the loss tensor (extracted from the trace output op) to `trace.backward()`.  
The call instruments the autograd engine and populates `grad_fns`, `saved_grad_fns`, and Op gradient fields.

In [ ]:
# Extract scalar loss from the last op (output_1)
loss = trace[trace.layer_labels[-1]].out
print("loss shape      :", loss.shape)
print("loss value      :", loss.item())
print("requires_grad   :", loss.requires_grad)

# Run deferred backward capture
trace.backward(loss)

print()
print("After backward:")
print("  has_gradients     :", trace.has_gradients)
print("  has_backward_pass :", trace.has_backward_pass)
print("  num_grad_fns      :", trace.num_grad_fns)
print("  num_saved_grad_fns:", trace.num_saved_grad_fns)

## 4. Gradient fields on `Op`

After backward, Ops with captured gradients expose `.grad`, `.has_grad`, `.has_saved_gradient`  
and the autograd linkage metadata `.grad_fn_class_name` / `.grad_fn_label`.

In [ ]:
# NOTE: after backward, trace['relu'] resolves to the GradFn record (backward node)
# because 'relu' also matches backward node labels. Use the full forward layer label
# (from trace.layer_labels) to reach the Op record. This is a naming-collision GAP
# flagged in the GAPs section at the bottom.
relu_label = trace.layer_labels[2]  # 'relu_1_2'
linear_label = trace.layer_labels[1]  # 'linear_1_1'

relu_op = trace[relu_label]
linear_op = trace[linear_label]

print("=== relu Op ===")
print("label              :", relu_label)
print("has_grad           :", relu_op.has_grad)
print("has_saved_gradient :", relu_op.has_saved_gradient)
print("grad_fn_class_name :", relu_op.grad_fn_class_name)  # ReluBackward0
print("grad_fn_label      :", relu_op.grad_fn_label)
print("grad shape         :", relu_op.grad.shape if relu_op.grad is not None else None)
print("grad               :", relu_op.grad)

print()
print("=== linear Op ===")
print("label              :", linear_label)
print("has_grad           :", linear_op.has_grad)
print("grad_fn_class_name :", linear_op.grad_fn_class_name)  # AddmmBackward0
print("grad shape         :", linear_op.grad.shape if linear_op.grad is not None else None)
print("grad               :", linear_op.grad)

## 5. `trace.grad_fns` — GradFn accessor

`trace.grad_fns` holds all discovered autograd nodes; `trace.saved_grad_fns` is the subset
whose gradient was actually captured and saved.

In [ ]:
print("All GradFn nodes:")
print(repr(trace.grad_fns))

print()
print("Saved GradFn nodes (grad payload available):")
print(repr(trace.saved_grad_fns))

## 6. `GradFn.__repr__` — key fields

A `GradFn` record links the autograd node back to its forward Op (`has_op`, `op_label`),  
its module context, and its position in the backward graph (`parents`, `children`, `order`).

In [ ]:
# Pick the relu backward node
relu_gfn_label = "relu_back_1_3"
if relu_gfn_label in trace.grad_fns:
    relu_gfn = trace.grad_fns[relu_gfn_label]
else:
    # Fallback: pick first saved GradFn that maps to relu
    relu_gfn = next(
        (gfn for gfn in trace.saved_grad_fns.values() if "relu" in gfn.label),
        list(trace.saved_grad_fns.values())[0],
    )

print("GradFn repr:")
print(repr(relu_gfn))

print()
print("Key fields:")
print("  label            :", relu_gfn.label)
print("  class_name       :", relu_gfn.class_name)
print("  has_op           :", relu_gfn.has_op)
print("  op_label         :", relu_gfn.op_label)
print("  module_address   :", relu_gfn.module_address)
print("  parents          :", relu_gfn.parents)
print("  children         :", relu_gfn.children)
print("  order            :", relu_gfn.order)
print("  num_calls        :", relu_gfn.num_calls)

## 7. `GradFnCall.__repr__` — per-fire records

Each time a GradFn fires (one per backward pass per node), a `GradFnCall` record is stored  
with the actual `grad_inputs` and `grad_outputs` tensor payloads.

In [ ]:
# Pick a saved GradFn with a real grad payload
saved_gfns = list(trace.saved_grad_fns.values())
gfn = saved_gfns[0]  # first saved node

print(f"GradFn: {gfn.label}  (class={gfn.class_name}, has_op={gfn.has_op})")
print()
print("Calls:")
print(repr(gfn.calls))

# Inspect the first call record
first_call = list(gfn.calls.values())[0]
print()
print("GradFnCall repr:")
print(repr(first_call))

print()
print("Key fields:")
print("  call_index        :", first_call.call_index)
print("  ordinal           :", first_call.ordinal)
print("  backward_pass_index:", first_call.backward_pass_index)
print("  grad_inputs       :", first_call.grad_inputs)
print("  grad_outputs      :", first_call.grad_outputs)
print("  is_saved          :", first_call.is_saved)

## 8. Iterating all saved GradFn records

Show grad shapes across the full backward graph — a quick summary of what was saved.

In [ ]:
print(f"{'GradFn label':<30} {'class':<22} {'has_op':<8} {'op_label':<15} {'grad_out shapes'}")
print("-" * 100)
for label, gfn in trace.saved_grad_fns.items():
    call = list(gfn.calls.values())[0]
    grad_shapes = str([tuple(g.shape) for g in call.grad_outputs if g is not None])
    print(
        f"{label:<30} {gfn.class_name:<22} {str(gfn.has_op):<8} {str(gfn.op_label or ''):<15} {grad_shapes}"
    )

## 9. `tl.validate` — forward, saved, and backward parity

`tl.validate` re-runs the model independently and checks that TorchLens' capture matches  
ground truth. Valid scopes: `'forward'`, `'saved'`, `'backward'`, `'intervention'`.

Returns `True` on pass (prints nothing by default); raises or returns `False` on failure.

In [ ]:
model_v, x_v = ZOO["linear_relu"]()

# Forward: checks that forward activations replay correctly
fwd_ok = tl.validate(model_v, x_v, scope="forward")
print(f"scope='forward'  : {fwd_ok}")

# Saved: checks that saved-out tensors match replay
saved_ok = tl.validate(model_v, x_v, scope="saved")
print(f"scope='saved'    : {saved_ok}")


# Backward: checks gradient parity; requires a scalar loss_fn
def loss_fn(out):
    """Return output as-is (linear_relu already returns a scalar sum)."""
    return out


bwd_ok = tl.validate(model_v, x_v, scope="backward", loss_fn=loss_fn)
print(f"scope='backward' : {bwd_ok}")

print()
print("All validation checks passed:", all([fwd_ok, saved_ok, bwd_ok]))

## 10. `trace.draw_backward()` — backward graph render smoke

`draw_backward()` renders the `GradFn` autograd graph to a PDF (or other format).  
Here we write to a temp file and confirm it is non-empty — no pixel inspection.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    bwd_path = os.path.join(tmpdir, "bwd_graph")
    # draw_backward returns the DOT string when vis_save_only=True
    _dot_src = trace.draw_backward(
        vis_outpath=bwd_path,
        vis_save_only=True,
        vis_fileformat="pdf",
        vis_mode="rolled",  # 'rolled' (default) or 'unrolled'
    )
    pdf_path = bwd_path + ".pdf"
    pdf_exists = os.path.exists(pdf_path)
    pdf_size = os.path.getsize(pdf_path) if pdf_exists else 0

print(f"draw_backward: pdf exists={pdf_exists}, size={pdf_size} B")
assert pdf_exists, "draw_backward did not produce a PDF file"
assert pdf_size > 0, "draw_backward produced an empty PDF"
print("draw_backward smoke: PASS")
print()
# Show a snippet of the returned DOT source
lines = _dot_src.strip().splitlines()[:6]
print("DOT source (first 6 lines):")
for ln in lines:
    print(" ", ln)

## 11. Second model — `view_model` (view/reshape in backward)

`ViewModel` applies `.view(2, 3, 2).sum()` — exercises ViewBackward in the backward graph.

In [ ]:
vm, vx = ZOO["view_model"]()

trace_v = tl.trace(vm, vx, backward_ready=True, save_grads=True)
loss_v = trace_v[trace_v.layer_labels[-1]].out
trace_v.backward(loss_v)

print(repr(trace_v))
print()
print(f"num_grad_fns      : {trace_v.num_grad_fns}")
print(f"num_saved_grad_fns: {trace_v.num_saved_grad_fns}")
print()
print("Saved GradFns:")
for label, gfn in trace_v.saved_grad_fns.items():
    print(f"  {label:<28} class={gfn.class_name}")

---

## ⚠️ GAPs / ergonomic smells

- **`trace['relu']` returns a `GradFn` not an `Op` after backward:** After calling
  `trace.backward()`, indexing by a bare function-type substring like `'relu'` resolves to the
  GradFn backward node (`relu_back_1_3`) instead of the forward Op (`relu_1_2`) because backward
  records are searched first. Users must use the full forward label (e.g. `trace['relu_1_2']` or
  `trace[trace.layer_labels[2]]`) to reach the Op post-backward. This is a significant ergonomic
  collision and should either be documented prominently or changed so forward Ops always take
  priority when the caller is clearly asking for a forward record.

- **`bwd_hook` callable contract is non-obvious:** The fn passed to `tl.bwd_hook(fn)` must accept
  a first positional grad argument AND a keyword-only `hook` argument: `def fn(g, *, hook): ...`.
  The error messages are opaque ("hook must accept a first positional grad argument" if the first
  positional is missing). This contract should be prominently documented.

- **`save_grads=True` alone is insufficient — `backward_ready=True` is also required:** A user
  passing only `save_grads=True` and calling `trace.backward(loss)` triggers an
  `AttributeError: 'GradFn' object has no attribute 'grad'` downstream. Both kwargs are needed;
  their interaction should be clearly documented.

- **`tl.validate(..., scope='saved_out')` raises `ValueError`:** Valid scopes are
  `['backward', 'forward', 'intervention', 'saved']`. The `'saved_out'` alias doesn't exist.

- **`trace.draw_backward()` returns a DOT string, not the output path:** `trace.draw()` returns
  the file path; `draw_backward()` returns the raw DOT string. Inconsistent return types.

- **`tl.validate` always returns `bool`:** No structured result (which layers failed, tolerances).
  A richer return would help when validating large models.